In [2]:
# feature engineering 

import pandas as pd

files = {"income_downpayment": r"C:\sta141B\Metro Homeowner Income Downpayment Data.csv",
    "housing_growth": r"C:\sta141B\Metro Housing Data SFR Condo Growth.csv",
    "zhvi_monthly_growth": r"C:\sta141B\Metro_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv"}

def feature_engineering_part(feature_engineeering):
    df = pd.read_csv(feature_engineeering)
    # remove extra spaces
    df.columns = df.columns.str.strip()
    # select available identity columns
    identity_columns = [c for c in ["RegionID", "SizeRank", "RegionName", "RegionType", "StateName", "BaseDate"] if c in df.columns]
    # convert to long format, and then convert to correct date format
    df = df.melt(id_vars=identity_columns, var_name="Date", value_name="Value")
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df["Value"] = pd.to_numeric(df["Value"], errors="coerce")
    # remove unavailable columns and sort in order by time and cities
    df = df.dropna(subset=["Date"]).sort_values(["RegionName", "Date"]).reset_index(drop=True)
    group = df.groupby("RegionName")["Value"]
    df["Year"] = df["Date"].dt.year
    df["Month"] = df["Date"].dt.month
    # shifting to previous other time points
    df["Lag1"] = group.shift(1)
    df["Lag12"] = group.shift(12)
    df["MoM"] = (df["Value"] - df["Lag1"]) / df["Lag1"]
    df["YoY"] = (df["Value"] - df["Lag12"]) / df["Lag12"]

    return df

results = {name: feature_engineering_part(feature_engineeering) for name, feature_engineeering in files.items()}

income_downpayment_feature_engineering = results["income_downpayment"]
zhvi_monthly_growth_feature_engineering = results["zhvi_monthly_growth"]

print("Income Downpayment Feature Engineering:")
print(income_downpayment_feature_engineering)
print("\n\n\n", "Zillow Home Value Index Growth(Variation) Feature Engineering:")
print(zhvi_monthly_growth_feature_engineering)


Income Downpayment Feature Engineering:
       RegionID  SizeRank   RegionName RegionType StateName       Date  \
0        394299       251  Abilene, TX        msa        TX 2012-01-31   
1        394299       251  Abilene, TX        msa        TX 2012-02-29   
2        394299       251  Abilene, TX        msa        TX 2012-03-31   
3        394299       251  Abilene, TX        msa        TX 2012-04-30   
4        394299       251  Abilene, TX        msa        TX 2012-05-31   
...         ...       ...          ...        ...       ...        ...   
65736    395247       225     Yuma, AZ        msa        AZ 2025-09-30   
65737    395247       225     Yuma, AZ        msa        AZ 2025-10-31   
65738    395247       225     Yuma, AZ        msa        AZ 2025-11-30   
65739    395247       225     Yuma, AZ        msa        AZ 2025-12-31   
65740    395247       225     Yuma, AZ        msa        AZ 2026-01-31   

             Value  Year  Month         Lag1        Lag12       MoM    